In [1]:
import numpy as np # operations mathematiques
import numpy.random as npr # simulation d ' alea
import matplotlib.pyplot as plt # plot
import math 
import tabata as tbt
import pandas as pd
import os

In [3]:
pip install anywidget


Note: you may need to restart the kernel to use updated packages.


$\textbf{Pour le premier avion}$

In [2]:
storename = r"C:\Users\yacqu\OneDrive\Bureau\Scolaire\TP\MACS2\TP Python\Aircraft_01.h5"
ds = tbt.Opset(storename) 

In [4]:
ds.df

,ALT [ft],EGT_1 [deg C],EGT_2 [deg C],FMV_1 [mm],FMV_2 [mm],HPTACC_1 [%],HPTACC_2 [%],M [Mach],N1_1 [% rpm],N1_2 [% rpm],...,VIB_AN2_1 [ips],VIB_AN2_2 [ips],VIB_BN1_1 [mils],VIB_BN1_2 [mils],VIB_BN2_1 [ips],VIB_BN2_2 [ips],VSV_1 [mm],VSV_2 [mm],CR,DEC
record_00,,,,,,,,,,,,,,,,,,,,,
0,-4.636641,33.494949,33.494949,-0.645530,-0.64913,4.637541,34.737062,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,56.592270,56.529748,False,False
1,-4.636641,33.494949,33.494949,-0.645530,-0.64913,4.637541,34.737062,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,56.592270,56.529748,False,False
2,-4.636641,33.494949,33.494949,-0.645530,-0.64913,4.696870,34.737062,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,56.592270,56.592270,False,False
3,-4.636641,33.494949,33.494949,-0.645530,-0.64913,4.696870,34.737062,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,56.592270,56.529748,False,False
4,-4.636641,33.494949,33.494949,-0.645530,-0.64913,4.696870,34.737062,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,56.592270,56.529748,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7424,180.828981,229.538918,239.390374,-0.795385,-0.79982,34.420641,34.737062,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,56.665211,56.665211,False,True
7425,180.828981,228.553773,238.405228,-0.795385,-0.79982,34.420641,34.796391,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,56.665211,56.665211,False,True
7426,180.828981,228.553773,238.405228,-0.795385,-0.79982,34.489858,34.855719,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,56.665211,56.665211,False,True


In [16]:
ancien_nom = "ALT [FT]"
nouveau_nom = "ALT"

indices_vol_supprimes = []

# serrt à supprimer les h5 créée, pour recommencer avec un fiichier vide
for f in ["clean.h5","decollage.h5","croisiere.h5"]:
    if os.path.exists(f):
        os.remove(f)
# création des Opset : conteneurs à Dataframe         
ds_clean = tbt.Opset("clean.h5")
ds_decollage = tbt.Opset("decollage.h5")
ds_croisiere = tbt.Opset("croisiere.h5")


# nettoyage des données 
for i, df in enumerate(ds):


    # 1. DataFrame vide ou une dimension nulle 
    if df.empty or df.shape[1] == 0:
        indices_vol_supprimes.append(i)
        continue

    # 2. met tt en majuscule 
    df.columns = [c.strip().upper() for c in df.columns]

    # 3. colonne altitude absente
    if ancien_nom not in df.columns:
        indices_vol_supprimes.append(i)
        continue

    alt = df[ancien_nom]

    # 4. s'il y a des  NaN
    if alt.isna().mean() > 0.2:            
        indices_vol_supprimes.append(i)
        continue

    # 5. signal quasi constant en utilisant l'écart type
    if alt.std() < 10:  
        indices_vol_supprimes.append(i)
        continue
    
    # 6. Si l'avion n'atteint jamais 2000 ft
    if alt.max() < 2000:
        indices_vol_supprimes.append(i)
        continue
        
    # 7. conversion ft en m
    df[ancien_nom] = alt * 0.3048

    #8. On remplace le nom ALT [FT] par ALT
    df.rename(columns={ancien_nom: nouveau_nom}, inplace=True)

    ds_clean.put(df)

print("Vols supprimés :", indices_vol_supprimes)
print("Nombre de vols propres :", len(ds_clean))

    #9. Création de la colonne CR
for df in ds_clean:   # pour savoir si l'avion est en phase de croisière si oui on met TRUE. 
    mx = max(df["ALT"])
    df["CR"] = (df["ALT"]>mx-150) #150 mètres 
    ds_clean.put(df)



Vols supprimés : [6, 7, 8, 9, 675]
Nombre de vols propres : 997


In [17]:
#décollage 
for i in range(len(ds_clean)):
    df = ds_clean[i]
    
    # 1. On calcule la variation d'altitude entre chaque point (la pente)
    # .diff() donne la différence entre le point actuel et le précédent
    pente = df["ALT"].diff()
    
    # 2. On cherche le premier point où ça monte 
    # On utilise un lissage sur la pente pour ignorer les petits sauts 
    pente_lissee = pente.rolling(window=20).mean()
    
    # On trouve le premier index où la pente devient positive et stable
    indices_montee = df.index[pente_lissee > 1] # Seuil de montée 
    
    if len(indices_montee) > 0:  #pour prendre en compte les vrais décollage 
        idx_debut = indices_montee[0]
        
        # 3. On extrait juste le "saut" : 50 points avant et 400 points après
        start = max(0, idx_debut - 50) 
        # on prend le min sert à s'assurer que le programme n'aille pas chercher une donnée en dehors du tableau df
        end = min(len(df), idx_debut + 400)
        
        df_dec = df.iloc[start:end]   
        ds_decollage.put(df_dec)
        
        
# phase de croisière
vols_extraits = []
for i in range(len(ds_clean)):
    df = ds_clean[i]
    
    #1. On cherche tous les lignes où l'altitude est dans la zone "CR"
    indices_croisiere = df.index[df["CR"] == True]

    #2. extraction des données 
    if len(indices_croisiere) > 0:
        # Le premier True est le début de la croisière
        idx_start = indices_croisiere[0]
        # Le dernier True est la fin de la croisière
        idx_end = indices_croisiere[-1]
        
        # On extrait la tranche
        df_phase = df.loc[idx_start:idx_end]
        ds_croisiere.put(df_phase)
        
print(f"Extraction terminée !")
print(f"Fichier décollage : {len(ds_decollage)} segments enregistrés.")
print(f"Fichier croisière : {len(ds_croisiere)} segments enregistrés.")        

Extraction terminée !
Fichier décollage : 997 segments enregistrés.
Fichier croisière : 997 segments enregistrés.


In [20]:
indices_croisiere

Index([2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018,
       ...
       3379, 3380, 3381, 3382, 3383, 3384, 3385, 3386, 3387, 3388],
      dtype='int64', name='record_999', length=1379)

In [18]:
ds_decollage.plot()

In [19]:
ds_croisiere.plot()

$\textbf{Pour le deuxième avion}$

In [2]:
storename = r"C:\Users\yacqu\OneDrive\Bureau\Scolaire\TP\MACS2\TP Python\Aircraft_02.h5"
ds2 = tbt.Opset(storename) 

In [6]:
ds2.plot()

In [4]:
ancien_nom = "ALT [FT]"
nouveau_nom = "ALT"

indices_vol_supprimes = []

# sert à supprimer les h5 créée, pour recommencer avec un fiichier vide
for f in ["clean2.h5","decollage2.h5","croisiere2.h5"]:
    if os.path.exists(f):
        os.remove(f)
# création des Opset : conteneurs à Dataframe         
ds2_clean = tbt.Opset("clean2.h5")
ds2_decollage = tbt.Opset("decollage2.h5")
ds2_croisiere = tbt.Opset("croisiere2.h5")


# nettoyage des données 
for i, df in enumerate(ds2):


    # 1. DataFrame vide ou une dimension nulle 
    if df.empty or df.shape[1] == 0:
        indices_vol_supprimes.append(i)
        continue

    # 2. met tt en majuscule 
    df.columns = [c.strip().upper() for c in df.columns]

    # 3. colonne altitude absente
    if ancien_nom not in df.columns:
        indices_vol_supprimes.append(i)
        continue

    alt = df[ancien_nom]

    # 4. s'il y a des  NaN
    if alt.isna().mean() > 0.2:            
        indices_vol_supprimes.append(i)
        continue

    # 5. signal quasi constant en utilisant l'écart type
    if alt.std() < 10:  
        indices_vol_supprimes.append(i)
        continue
    
    # 6. Si l'avion n'atteint jamais 2000 ft
    if alt.max() < 2000:
        indices_vol_supprimes.append(i)
        continue
        
    # 7. conversion ft en m
    df[ancien_nom] = alt * 0.3048

    #8. On remplace le nom ALT [FT] par ALT
    df.rename(columns={ancien_nom: nouveau_nom}, inplace=True)

    ds2_clean.put(df)

print("Vols supprimés :", indices_vol_supprimes)
print("Nombre de vols propres :", len(ds2_clean))

    #9. Création de la colonne CR
for df in ds2_clean:   # pour savoir si l'avion est en phase de croisière si oui on met TRUE. 
    mx = max(df["ALT"])
    df["CR"] = (df["ALT"]>mx-150) #150 mètres 
    ds2_clean.put(df)


Vols supprimés : [700]
Nombre de vols propres : 1001


In [7]:
ds2_clean.plot()

In [8]:
#décollage 
for i in range(len(ds2_clean)):
    df = ds2_clean[i]
    
    # 1. On calcule la variation d'altitude entre chaque point (la pente)
    # .diff() donne la différence entre le point actuel et le précédent
    pente = df["ALT"].diff()
    
    # 2. On cherche le premier point où ça monte 
    # On utilise un lissage sur la pente pour ignorer les petits sauts 
    pente_lissee = pente.rolling(window=20).mean()
    
    # On trouve le premier index où la pente devient positive et stable
    indices_montee = df.index[pente_lissee > 1] # Seuil de montée 
    
    if len(indices_montee) > 0:  #pour prendre en compte les vrais décollage 
        idx_debut = indices_montee[0]
        
        # 3. On extrait juste le "saut" : 50 points avant et 400 points après
        start = max(0, idx_debut - 50) 
        # on prend le min sert à s'assurer que le programme n'aille pas chercher une donnée en dehors du tableau df
        end = min(len(df), idx_debut + 400)
        
        df_dec = df.iloc[start:end]   
        ds2_decollage.put(df_dec)
        
        
# phase de croisière
vols_extraits = []
for i in range(len(ds2_clean)):
    df = ds2_clean[i]
    
    #1. On cherche tous les lignes où l'altitude est dans la zone "CR"
    indices_croisiere = df.index[df["CR"] == True]

    #2. extraction des données 
    if len(indices_croisiere) > 0:
        # Le premier True est le début de la croisière
        idx_start = indices_croisiere[0]
        # Le dernier True est la fin de la croisière
        idx_end = indices_croisiere[-1]
        
        # On extrait la tranche
        df_phase = df.loc[idx_start:idx_end]
        ds2_croisiere.put(df_phase)
        
print(f"Extraction terminée !")
print(f"Fichier décollage : {len(ds2_decollage)} segments enregistrés.")
print(f"Fichier croisière : {len(ds2_croisiere)} segments enregistrés.")        

Extraction terminée !
Fichier décollage : 1001 segments enregistrés.
Fichier croisière : 1001 segments enregistrés.


In [10]:
ds2_decollage.plot()

$\textbf{Pour le trisième avion}$

In [14]:
storename = r"C:\Users\yacqu\OneDrive\Bureau\Scolaire\TP\MACS2\TP Python\Aircraft_03.h5"
ds3 = tbt.Opset(storename) 

In [15]:
ds3.plot()

In [16]:
ancien_nom = "ALT [FT]"
nouveau_nom = "ALT"

indices_vol_supprimes = []

# sert à supprimer les h5 créée, pour recommencer avec un fiichier vide
for f in ["clean3.h5","decollage3.h5","croisiere3.h5"]:
    if os.path.exists(f):
        os.remove(f)
# création des Opset : conteneurs à Dataframe         
ds3_clean = tbt.Opset("clean3.h5")
ds3_decollage = tbt.Opset("decollage3.h5")
ds3_croisiere = tbt.Opset("croisiere3.h5")


# nettoyage des données 
for i, df in enumerate(ds3):


    # 1. DataFrame vide ou une dimension nulle 
    if df.empty or df.shape[1] == 0:
        indices_vol_supprimes.append(i)
        continue

    # 2. met tt en majuscule 
    df.columns = [c.strip().upper() for c in df.columns]

    # 3. colonne altitude absente
    if ancien_nom not in df.columns:
        indices_vol_supprimes.append(i)
        continue

    alt = df[ancien_nom]

    # 4. s'il y a des  NaN
    if alt.isna().mean() > 0.2:            
        indices_vol_supprimes.append(i)
        continue

    # 5. signal quasi constant en utilisant l'écart type
    if alt.std() < 10:  
        indices_vol_supprimes.append(i)
        continue
    
    # 6. Si l'avion n'atteint jamais 2000 ft
    if alt.max() < 2000:
        indices_vol_supprimes.append(i)
        continue
        
    # 7. conversion ft en m
    df[ancien_nom] = alt * 0.3048

    #8. On remplace le nom ALT [FT] par ALT
    df.rename(columns={ancien_nom: nouveau_nom}, inplace=True)

    ds3_clean.put(df)

print("Vols supprimés :", indices_vol_supprimes)
print("Nombre de vols propres :", len(ds3_clean))

    #9. Création de la colonne CR
for df in ds3_clean:   # pour savoir si l'avion est en phase de croisière si oui on met TRUE. 
    mx = max(df["ALT"])
    df["CR"] = (df["ALT"]>mx-150) #150 mètres 
    ds3_clean.put(df)


Vols supprimés : [417, 904]
Nombre de vols propres : 1000


In [19]:
#décollage 
for i in range(len(ds3_clean)):
    df = ds3_clean[i]
    
    # 1. On calcule la variation d'altitude entre chaque point (la pente)
    # .diff() donne la différence entre le point actuel et le précédent
    pente = df["ALT"].diff()
    
    # 2. On cherche le premier point où ça monte 
    # On utilise un lissage sur la pente pour ignorer les petits sauts 
    pente_lissee = pente.rolling(window=20).mean()
    
    # On trouve le premier index où la pente devient positive et stable
    indices_montee = df.index[pente_lissee > 1] # Seuil de montée 
    
    if len(indices_montee) > 0:  #pour prendre en compte les vrais décollage 
        idx_debut = indices_montee[0]
        
        # 3. On extrait juste le "saut" : 50 points avant et 400 points après
        start = max(0, idx_debut - 50) 
        # on prend le min sert à s'assurer que le programme n'aille pas chercher une donnée en dehors du tableau df
        end = min(len(df), idx_debut + 400)
        
        df_dec = df.iloc[start:end]   
        ds3_decollage.put(df_dec)
        
        
# phase de croisière
vols_extraits = []
for i in range(len(ds3_clean)):
    df = ds3_clean[i]
    
    #1. On cherche tous les lignes où l'altitude est dans la zone "CR"
    indices_croisiere = df.index[df["CR"] == True]

    #2. extraction des données 
    if len(indices_croisiere) > 0:
        # Le premier True est le début de la croisière
        idx_start = indices_croisiere[0]
        # Le dernier True est la fin de la croisière
        idx_end = indices_croisiere[-1]
        
        # On extrait la tranche
        df_phase = df.loc[idx_start:idx_end]
        ds3_croisiere.put(df_phase)
        
print(f"Extraction terminée !")
print(f"Fichier décollage : {len(ds3_decollage)} segments enregistrés.")
print(f"Fichier croisière : {len(ds3_croisiere)} segments enregistrés.")        

Extraction terminée !
Fichier décollage : 1000 segments enregistrés.
Fichier croisière : 1000 segments enregistrés.


In [21]:
ds3_croisiere.plot()